<a href="https://colab.research.google.com/github/Arobnett/HDX-sources-and-more-API-connection/blob/main/notebooks/00_main_run_all_open_source.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 Main Run All Open Source

Runs the validated public-source pipeline and downloads a timestamped package containing Bronze, Silver, Gold, validation, EDA reports, and EDA visuals.

**Public-data boundary:** this notebook does not use restricted target variables or internal operational datasets.


In [ ]:
from pathlib import Path  # Work with repository paths.

REPO_URL = "https://github.com/Arobnett/HDX-sources-and-more-API-connection.git"  # Public pipeline repository.
REPO_DIR = Path("/content/HDX-sources-and-more-API-connection")  # Colab checkout location.

%cd /content

if not REPO_DIR.exists():  # Clone the repository in a fresh runtime.
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/HDX-sources-and-more-API-connection

!git checkout main  # Use the validated default branch.
!git pull origin main  # Refresh the local checkout.
!git lfs install  # Enable large-file support.
!git lfs pull  # Retrieve Git LFS-managed public inputs.
!pip install -q pycountry  # Install the ISO country reference used by geographic validation.


In [ ]:
import importlib.util  # Load repository modules directly from source files.
import sys  # Register loaded modules for cross-module imports.
from pathlib import Path  # Work with filesystem paths.

PROJECT_ROOT = Path("/content/HDX-sources-and-more-API-connection")  # Set repository root.
SRC_DIR = PROJECT_ROOT / "src"  # Set reusable source-code directory.

def load_module(module_name, module_path):  # Load one repository Python module.
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

paths = load_module("paths", SRC_DIR / "paths.py")  # Load canonical paths first.
cleaning = load_module("cleaning", SRC_DIR / "cleaning.py")  # Load Silver cleaning.
geographic_keys = load_module("geographic_keys", SRC_DIR / "geographic_keys.py")  # Load geographic QA.
feature_assembly = load_module("feature_assembly", SRC_DIR / "feature_assembly.py")  # Load Gold assembly.
eda_reporting = load_module("eda_reporting", SRC_DIR / "eda_reporting.py")  # Load expanded EDA reporting.

CLEAN_DIR = paths.CLEAN_DIR
CLEAN_REPORTS_DIR = paths.CLEAN_REPORTS_DIR
FEATURE_REPORTS_DIR = paths.FEATURE_REPORTS_DIR
MODEL_FEATURES_DIR = paths.MODEL_FEATURES_DIR
EDA_REPORTS_DIR = paths.EDA_REPORTS_DIR
EDA_VISUALS_DIR = paths.OUTPUTS_DIR / "eda_visuals"

print(f"Project root: {PROJECT_ROOT}")
print(f"Standardized source directory: {paths.SILVER_DIR}")


In [ ]:
# STEP 1 - Clean standardized public-source files into model-ready Silver tables.
silver_summary, silver_rejects = cleaning.clean_silver_directory()
display(silver_summary)

silver_errors = silver_summary[silver_summary["status"].eq("error")]
if not silver_errors.empty:
    raise RuntimeError("Silver cleaning contains failed sources.")

print(f"Clean Silver files: {len(list(CLEAN_DIR.glob('*.csv')))}")


In [ ]:
# STEP 2 - Enforce canonical geographic keys before cross-source joins.
geo_summary = geographic_keys.validate_clean_directory(CLEAN_DIR, CLEAN_REPORTS_DIR)
display(geo_summary)

if geo_summary.empty:
    raise RuntimeError("Geographic validation returned no Silver sources.")

if not geo_summary["key_unique"].all():
    raise RuntimeError("At least one Silver source failed geographic-key uniqueness.")

print("Silver geographic validation passed.")


In [ ]:
# STEP 3 - Assemble the validated public Gold feature table.
gold_features, assembly_report, feature_catalog = feature_assembly.assemble_feature_table()

display(assembly_report)

KEY_COLUMNS = ["iso3", "country", "year", "month"]
duplicate_gold_keys = int(gold_features.duplicated(KEY_COLUMNS).sum())

if gold_features.empty:
    raise RuntimeError("Gold feature table is empty.")

if duplicate_gold_keys:
    raise RuntimeError(f"Gold contains {duplicate_gold_keys} duplicate modeling keys.")

print(f"Gold rows: {len(gold_features):,}")
print(f"Gold columns: {len(gold_features.columns):,}")
print(f"Duplicate Gold keys: {duplicate_gold_keys:,}")


In [ ]:
# STEP 4 - Generate all public-feature EDA reports and reusable PNG visuals.
eda_outputs = eda_reporting.generate_eda_outputs(
    gold_features,
    EDA_REPORTS_DIR,
    EDA_VISUALS_DIR,
)

print("\nEDA report files:")
for path in sorted(EDA_REPORTS_DIR.glob("*")):
    print(f" - {path.name}")

print("\nEDA visual files:")
for path in sorted(EDA_VISUALS_DIR.glob("*.png")):
    print(f" - {path.name}")

expected_visuals = {
    "gold_integrity_summary.png",
    "feature_missingness.png",
    "temporal_coverage.png",
    "geographic_coverage.png",
    "numeric_distributions.png",
    "pairwise_feature_correlations.png",
    "eda_findings_summary.png",
}

missing_visuals = expected_visuals - {path.name for path in EDA_VISUALS_DIR.glob("*.png")}
if missing_visuals:
    raise RuntimeError(f"Missing expected EDA visuals: {sorted(missing_visuals)}")

print(f"Expanded EDA validation passed: {len(expected_visuals)} expected visuals created.")


In [ ]:
# STEP 5 - Package the public pipeline outputs into short, SharePoint/Windows-friendly folders.
import shutil  # Copy output folders and create the ZIP archive.
from datetime import datetime, timezone  # Timestamp each exported pipeline run.

EXPORT_ROOT = Path("/content/public_crisis_risk_pipeline_export")  # Build a temporary export tree.
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
EXPORT_ROOT.mkdir(parents=True)

def copy_folder(source, destination):  # Copy one generated output folder when present.
    destination.mkdir(parents=True, exist_ok=True)
    if source.exists():
        for item in source.iterdir():
            target = destination / item.name
            if item.is_dir():
                shutil.copytree(item, target, dirs_exist_ok=True)
            else:
                shutil.copy2(item, target)

copy_folder(PROJECT_ROOT / "country_month_year_outputs", EXPORT_ROOT / "01_bronze")
copy_folder(CLEAN_DIR, EXPORT_ROOT / "02_silver")
copy_folder(MODEL_FEATURES_DIR, EXPORT_ROOT / "03_gold")
copy_folder(FEATURE_REPORTS_DIR, EXPORT_ROOT / "04_validation")
copy_folder(CLEAN_REPORTS_DIR, EXPORT_ROOT / "04_validation")
copy_folder(EDA_REPORTS_DIR, EXPORT_ROOT / "05_eda")
copy_folder(EDA_VISUALS_DIR, EXPORT_ROOT / "06_visuals")

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
zip_base = Path("/content") / f"public_crisis_risk_pipeline_{timestamp}"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", EXPORT_ROOT))

print(f"Export ZIP: {zip_path}")
print(f"EDA visuals packaged: {len(list((EXPORT_ROOT / '06_visuals').glob('*.png')))}")


In [ ]:
# STEP 6 - Download the complete public pipeline package.
from google.colab import files  # Use Colab's browser download helper.

files.download(str(zip_path))
